# Two Worlds Studio — независимая проверка CPU и GPU (Kaggle)

Этот Notebook запускает **сам владелец аккаунта Kaggle** в своём сеансе. Он не запрашивает API-токены, не обращается к чужому аккаунту, не скачивает веса моделей и не запускает генерацию видео.

Перед запуском владелец выбирает Accelerator = None (проверка CPU) или GPU (проверка CPU и маленькое CUDA-вычисление). Проверка GPU потребляет небольшое время GPU-сессии Kaggle. Успех проверки **не гарантирует**, что Wan2.2 или иная большая модель поместится в видеопамять.

Файл результата: two_worlds_compute_preflight.json (Kaggle Output). Сохраняйте его приватно; для передачи достаточно JSON, без паролей и секретов.

In [ ]:
# Владелец Kaggle выполняет этот блок вручную. Никаких API-токенов, сетевых запросов и весов моделей.
import hashlib
import json
import os
import shutil
import sys
import time
from pathlib import Path

def gib(value):
    return round(value / (1024 ** 3), 2)

def system_memory_gib():
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemTotal:"):
                return gib(int(line.split()[1]) * 1024)
    except (OSError, ValueError, IndexError):
        pass
    return None

start = time.monotonic()
digest = hashlib.sha256(b"two-worlds-cpu-check" * 16384).hexdigest()
cpu = {
    "logical_cores": os.cpu_count(),
    "ram_gib": system_memory_gib(),
    "disk_free_gib": gib(shutil.disk_usage("/kaggle/working" if Path("/kaggle/working").is_dir() else ".").free),
    "smoke": "PASS" if len(digest) == 64 else "FAIL",
    "smoke_sha256": digest,
    "smoke_ms": max(0, round((time.monotonic() - start) * 1000, 2)),
}

gpu = {"torch_installed": False, "cuda_available": False, "device_count": 0,
       "devices": [], "smoke": "SKIPPED"}
try:
    import torch
    gpu["torch_installed"] = True
    gpu["cuda_available"] = bool(torch.cuda.is_available())
    if gpu["cuda_available"]:
        count = int(torch.cuda.device_count())
        gpu["device_count"] = count
        for i in range(count):
            p = torch.cuda.get_device_properties(i)
            gpu["devices"].append({"name": str(p.name)[:100],
                                   "vram_gib": gib(p.total_memory)})
        if count:
            try:
                x = torch.ones((256, 256), dtype=torch.float32, device="cuda:0")
                y = x @ x
                torch.cuda.synchronize(0)
                gpu["smoke"] = "PASS" if bool(torch.isfinite(y).all().item()) else "FAIL"
                del x, y
            except Exception:
                gpu["smoke"] = "FAIL"
except Exception:
    gpu["smoke"] = "FAIL" if gpu["torch_installed"] else "SKIPPED"

report = {
    "schema": 1,
    "source": "owner_operated_kaggle_notebook",
    "python": ".".join(map(str, sys.version_info[:3])),
    "cpu": cpu,
    "gpu": gpu,
    "model_inference_tested": False,
    "weights_downloaded": False,
    "credentials_used": False,
    "video_generated": False,
}
out_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
output = out_dir / "two_worlds_compute_preflight.json"
output.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Receipt saved:", output.name)
